# Teste isolado — ARVAP (Agência de Regulação, Controle e Fiscalização do Vale do Paranapanema)

Fonte candidata: ARVAP, setor Saneamento (regional — municípios do Vale do
Paranapanema/SP). Notebook **descartável** (Fase 1) — sem dispatcher, sem
`atualizar_status_fonte`, sem gravar nada. Só valida:

1. Parsing do feed RSS de categoria (título, data, link)
2. Se o texto embutido no próprio feed (`content:encoded`) já é o artigo
   completo, ou se ainda precisa baixar a página individual

## Confirmado antes de assumir

WordPress confirmado (plugin "WordPress Download Manager", `wp-content`
padrão). **Achado 1**: `/noticias/feed/` (a URL óbvia) é o feed de
**comentários** da página, sempre vazio — WordPress trata `/pagina/feed/`
como comentários por padrão, só devolve posts quando aplicado a uma
categoria/tag. O feed certo é `/category/noticias/feed/`.

**Achado 2**: o feed de categoria já vem com `<content:encoded>` contendo o
HTML completo do artigo, embutido — não é só resumo. Isso significa que,
em teoria, não precisaríamos baixar a página do artigo separadamente; mas
testo os dois caminhos aqui pra decidir na Fase 2, e principalmente pra
confirmar que **todo item tem `pubDate` válido** (foi item sem data — não
tamanho de payload — que causou o 422 no PSR/Exame; ver Changelog).

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml feedparser
dbutils.library.restartPython()


In [0]:
import re
import time
import random
from typing import Optional

import httpx
import feedparser
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests


In [0]:
# =============================================================================
# Configuração
# =============================================================================

FEED_URL = "https://arvap.sp.gov.br/category/noticias/feed/"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")


In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer="https://arvap.sp.gov.br/")
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


## Teste 1 — parsear o feed e checar integridade

Cada item precisa ter `title`, `link`, `published_parsed` válido. Item sem
data é exatamente o padrão que causou o 422 no PSR/Exame — checando aqui
antes de integrar, não depois.

In [0]:
resp_feed = httpx.get(FEED_URL, headers=headers_aleatorios(), timeout=HTTP_TIMEOUT, follow_redirects=True)
print(f"status={resp_feed.status_code}, tamanho={len(resp_feed.content)} bytes")

parsed = feedparser.parse(resp_feed.content)
print(f"bozo={parsed.bozo}")
print(f"{len(parsed.entries)} itens no feed.\n")

itens = []
sem_data = []
for entry in parsed.entries:
    titulo = entry.get("title", "")
    link = entry.get("link", "")
    tem_data = bool(entry.get("published_parsed"))
    tem_conteudo_embutido = bool(entry.get("content"))

    if not tem_data:
        sem_data.append(titulo)

    itens.append({
        "titulo": titulo,
        "url": link,
        "published_parsed": entry.get("published_parsed"),
        "tem_conteudo_embutido": tem_conteudo_embutido,
        "conteudo_embutido": entry.get("content", [{}])[0].get("value", "") if tem_conteudo_embutido else "",
    })

print(f"{'DATA':<12} {'CONTEÚDO?':<10} TÍTULO")
print("-" * 100)
for item in itens:
    data_str = time.strftime("%Y-%m-%d", item["published_parsed"]) if item["published_parsed"] else "SEM DATA"
    conteudo_str = f"{len(item['conteudo_embutido'])} chars" if item["tem_conteudo_embutido"] else "não"
    print(f"{data_str:<12} {conteudo_str:<10} {item['titulo'][:70]}")

print(f"\nItens sem data: {len(sem_data)}")
if sem_data:
    print("ATENÇÃO — títulos sem data:", sem_data)


## Teste 2 — comparar conteúdo embutido no feed vs. baixar a página do artigo

Confirma se dá pra confiar no `content:encoded` do feed (mais rápido, um
request a menos por matéria) ou se ainda precisa do fallback de baixar a
página individual, igual ao resto do Dispatcher 1.

In [0]:
def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "iframe", "form"]):
        tag.decompose()

    base = soup.select_one(".entry-content")
    if base is None or len(base.get_text(strip=True)) < 200:
        base = soup

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def limpar_html_embutido(html_bruto: str) -> str:
    soup = BeautifulSoup(html_bruto, "lxml")
    texto = soup.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


AMOSTRA = 5
comparacoes = []

for item in itens[:AMOSTRA]:
    print(f"\n  [item] {item['titulo'][:80]}")

    texto_embutido = limpar_html_embutido(item["conteudo_embutido"]) if item["tem_conteudo_embutido"] else ""
    print(f"    -> texto embutido no feed: {len(texto_embutido)} chars")

    html_pagina = baixar_pagina(item["url"])
    texto_pagina = extrair_texto_generico(html_pagina) if html_pagina else ""
    print(f"    -> texto baixando a página: {len(texto_pagina)} chars")

    comparacoes.append({
        "titulo": item["titulo"],
        "chars_embutido": len(texto_embutido),
        "chars_pagina": len(texto_pagina),
        "bate": abs(len(texto_embutido) - len(texto_pagina)) < 200,
    })

    time.sleep(random.uniform(0.5, 1.2))

print("\n" + "=" * 100)
for c in comparacoes:
    status = "OK, bate" if c["bate"] else "DIVERGE"
    print(f"{status:<10} embutido={c['chars_embutido']:>6}  pagina={c['chars_pagina']:>6}  {c['titulo'][:60]}")


In [0]:
detalhe = comparacoes[0]
titulo_completo = itens[0]["titulo"]
texto_completo = limpar_html_embutido(itens[0]["conteudo_embutido"])

print("=" * 100)
print(f"TÍTULO      : {titulo_completo}")
print(f"URL         : {itens[0]['url']}")
print(f"TAMANHO     : {len(texto_completo)} chars (via conteúdo embutido no feed)")
print("=" * 100)
print(texto_completo)


## Conclusão da Fase 1

Confirmar aqui, depois de rodar: (a) se `sem_data` veio vazio — feed
consistente, sem risco do tipo 422 do PSR; (b) se `embutido` e `pagina`
batem — nesse caso, dá pra usar o `content:encoded` do feed direto,
economizando um request por matéria, algo que o Dispatcher 1 hoje não faz
(sempre baixa a página).

**Se os dois pontos confirmarem**: ARVAP entra no `ingest-news-rss-generico`
como está, sem mudança de arquitetura — só uma entrada em `CONFIGS_FONTES`
apontando pro feed de categoria. Não precisa de scraping (Dispatcher 3),
diferente da ABAR.